# Gaussian Process Regression

Consider the following [data set](https://www.kaggle.com/datasets/elikplim/eergy-efficiency-dataset) that has been created in an energy analysis using 12 different building shapes simulated in Ecotect. The buildings differ with respect to the glazing area, the glazing area distribution, and the orientation, amongst other parameters. The dataset contains eight attributes (or features, denoted by X1 to X8) and two responses (denoted by Y1 and Y2). Explore the possibility of modeling the 'heating load' and the 'cooling load' as a single parameter Gaussian process. Discuss your conclusions.

In [ ]:
import kagglehub

# Download latest version
kagglepath="elikplim/eergy-efficiency-dataset"
path = kagglehub.dataset_download(kagglepath)

print("Path to dataset files:", path)

In [ ]:
import os
print(f"Listing contents of: {path}")
!ls {path}
df2=pd.read_csv(path+"/ENB2012_data.csv")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.metrics import mean_squared_error, r2_score

# The starter code assigned the dataset to df2
# Let's drop any potential trailing NaN rows that this specific dataset is known to have
df_energy = df2.dropna()

# Define features (X1-X8) and targets (Y1, Y2)
X = df_energy[['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8']]
y_heat = df_energy['Y1']
y_cool = df_energy['Y2']

# It is highly recommended to scale features for GPR to ensure stable convergence
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, yh_train, yh_test, yc_train, yc_test = train_test_split(
    X_scaled, y_heat, y_cool, test_size=0.2, random_state=42
)

# Define a single-parameter (isotropic) RBF kernel
# 1.0 * RBF(1.0) creates a kernel with a constant variance and a single length-scale
kernel = C(1.0, (1e-3, 1e3)) * RBF(1.0, (1e-2, 1e2))

print("Training GPR for Heating Load (Y1)...")
gp_heat = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, random_state=42)
gp_heat.fit(X_train, yh_train)
yh_pred, yh_std = gp_heat.predict(X_test, return_std=True)

print("Training GPR for Cooling Load (Y2)...")
gp_cool = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, random_state=42)
gp_cool.fit(X_train, yc_train)
yc_pred, yc_std = gp_cool.predict(X_test, return_std=True)

# Evaluation
print("\n--- Model Performance ---")
print(f"Heating Load - R2 Score: {r2_score(yh_test, yh_pred):.4f}, MSE: {mean_squared_error(yh_test, yh_pred):.4f}")
print(f"Cooling Load - R2 Score: {r2_score(yc_test, yc_pred):.4f}, MSE: {mean_squared_error(yc_test, yc_pred):.4f}")
print("\nLearned Kernel for Heating:", gp_heat.kernel_)
print("Learned Kernel for Cooling:", gp_cool.kernel_)

Discussion for the Conclusion

Performance: The $R^2$ score tells you how well the model explains the variance. GPR generally performs exceptionally well on this specific dataset because it is relatively small and has highly non-linear, continuous spatial relationships.

The "Single Parameter" Constraint: By using an isotropic RBF kernel, we forced the model to assume that all 8 architectural features operate on the exact same length scale.

Limitation: In reality, features like "Glazing Area" and "Orientation" have completely different scales and impacts. A single parameter process handles this adequately only because we strictly scaled the data (StandardScaler) beforehand. Without standardizing the data, a single-parameter GP would fail to model the loads accurately.

# Linear Regression

Consider the following [data set](https://www.kaggle.com/datasets/programmer3/green-building-multi-source-environment-dataset). This dataset has 2400 samples provides a comprehensive collection of multi-source building environment data designed to support research in green building design, energy efficiency optimization, and indoor comfort prediction using advanced machine learning and deep learning techniques. Explore the possibility of predicting the 'predicted_energy_demand'  using a linear relationship of a suitable set of other data parameters. Justify your choice of parameters and discuss the results.

In [ ]:
import kagglehub

# Download latest version
kagglepath="programmer3/green-building-multi-source-environment-dataset" #"ujjwalchowdhury/energy-efficiency-data-set"
path = kagglehub.dataset_download(kagglepath)

print("Path to dataset files:", path)

In [ ]:
import os
print(f"Listing contents of: {path}")
!ls {path}
df2=pd.read_csv(path+"/green_building_dataset.csv")
inspector.df=df2

In [ ]:
import seaborn as sns
from sklearn.linear_model import LinearRegression

# The starter code overwrote df2 with the new dataset
df_green = df2.copy()

# 1. Feature Selection: Calculate correlations with the target variable
target = 'predicted_energy_demand'

# Filter only numeric columns for correlation
numeric_cols = df_green.select_dtypes(include=[np.number])
correlations = numeric_cols.corr()[target].sort_values(ascending=False)

print("Correlation with Predicted Energy Demand:")
print(correlations)

# Justification Step: Select features with a meaningful correlation (e.g., > 0.1 or < -0.1)
# You can manually list columns, but this automates the selection based on correlation strength.
threshold = 0.1
selected_features = correlations[abs(correlations) > threshold].index.tolist()
selected_features.remove(target) # Remove the target itself from the feature list

print(f"\nSelected Features based on absolute correlation > {threshold}:")
print(selected_features)

# 2. Prepare Data
X_lr = numeric_cols[selected_features]
y_lr = numeric_cols[target]

# Split data
X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(
    X_lr, y_lr, test_size=0.2, random_state=42
)

# 3. Train Linear Regression Model
lr_model = LinearRegression()
lr_model.fit(X_train_lr, y_train_lr)

# 4. Predict and Evaluate
y_pred_lr = lr_model.predict(X_test_lr)

print("\n--- Linear Regression Performance ---")
print(f"R2 Score: {r2_score(y_test_lr, y_pred_lr):.4f}")
print(f"Mean Squared Error: {mean_squared_error(y_test_lr, y_pred_lr):.4f}")

# Displaying Coefficients to understand the relationship
coef_df = pd.DataFrame({'Feature': selected_features, 'Coefficient': lr_model.coef_})
print("\nModel Coefficients:")
print(coef_df.sort_values(by='Coefficient', ascending=False))

Discussion and Justification Points

Justification of Parameters: In your report, state that you used Pearson Correlation to select your features. Explain that you dropped variables with near-zero correlation because they act as noise in a linear model and do not provide meaningful predictive power for energy demand.

Results Discussion: Look at the $R^2$ score printed by the code.

If the $R^2$ is high (>0.7): The linear model successfully captured the relationship, meaning building parameters influence energy demand in a mostly linear, additive way.

If the $R^2$ is low: Conclude that the relationship between the green building metrics and energy demand is highly non-linear or relies on interactions between variables (like how insulation interacts with outside temperature), which a standard linear regression cannot mathematically capture without feature engineering.

Coefficient Analysis: Briefly mention the coefficients outputted by the code. A positive coefficient means an increase in that feature increases energy demand; a negative coefficient means it decreases demand (improves efficiency).